# Software-Defined FM Receiver — Python & Google Colab

**Simulação de um transmissor e receptor FM em banda-base complexa (IQ).**

Este notebook implementa modulação FM, canal AWGN, demodulação por diferença de fase, filtragem Butterworth, análise espectral, waterfall, painel interativo e métricas de reconstrução.

> **Escopo:** simulação em software, sem recepção de RF real. O notebook não requer hardware SDR. Para reproduzir os resultados com voz, carregue seus próprios arquivos de áudio; nenhum áudio pessoal é distribuído neste repositório.

**Como executar:** abra no Google Colab e use **Ambiente de execução → Executar tudo**. A demonstração inicial usa áudio sintético. O carregamento de voz é opcional.

## 1. Bibliotecas e parâmetros

O sinal FM é representado por $s[n]=e^{j\phi[n]}$, com $\phi[n]=2\pi\Delta f\,\sum_{k=0}^{n}m[k]/f_s$. A taxa de amostragem é 16 kHz, o desvio nominal é 3 kHz e a mensagem é normalizada para amplitude de pico 0,7.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, spectrogram, welch
from IPython.display import Audio, display

FS = 16_000
DESVIO_HZ = 3_000
FC_REFERENCIA_HZ = 1_100
SEMENTE = 42

print(f"Amostragem: {FS} Hz | Desvio FM: {DESVIO_HZ} Hz")

## 2. Sinal de demonstração

Geramos três notas sucessivas (440 Hz, 523,25 Hz e 659,25 Hz). Este sinal permite executar o notebook inteiro sem uploads.

In [ ]:
def gerar_audio_demo(fs=FS):
    notas = [440.0, 523.25, 659.25]
    segmentos = []
    for freq in notas:
        t = np.arange(fs) / fs
        segmentos.append(0.7 * np.sin(2 * np.pi * freq * t))
    return np.concatenate(segmentos)

mensagem = gerar_audio_demo()
print(f"Duração do sinal: {len(mensagem)/FS:.2f} s")
display(Audio(mensagem, rate=FS, normalize=False))

## 3. Funções do transmissor, canal e receptor

O canal acrescenta ruído gaussiano complexo circular. O demodulador calcula a diferença de fase entre amostras consecutivas; por isso, a referência de áudio é alinhada usando `mensagem[1:]`.

**Limitação numérica:** em baixa SNR, erros de fase e envoltória muito pequena podem provocar grandes distorções. O modelo não inclui sincronização, pré-ênfase/de-ênfase ou front-end RF.

In [ ]:
def modular_fm(audio, fs=FS, desvio_hz=DESVIO_HZ):
    audio = np.asarray(audio, dtype=float)
    fase = 2 * np.pi * desvio_hz * np.cumsum(audio) / fs
    return np.exp(1j * fase)


def canal_awgn(tx, snr_db, rng):
    potencia_sinal = np.mean(np.abs(tx)**2)
    potencia_ruido = potencia_sinal / (10 ** (snr_db / 10))
    ruido = np.sqrt(potencia_ruido / 2) * (
        rng.standard_normal(len(tx)) + 1j * rng.standard_normal(len(tx))
    )
    return tx + ruido


def demodular_fm(rx, fs=FS, desvio_hz=DESVIO_HZ):
    delta_fase = np.angle(rx[1:] * np.conj(rx[:-1]))
    return delta_fase * fs / (2 * np.pi * desvio_hz)


def filtrar_audio(audio, fc_hz, fs=FS):
    if not 0 < fc_hz < fs / 2:
        raise ValueError("A frequência de corte deve estar entre 0 e fs/2")
    sos = butter(4, fc_hz, fs=fs, btype="low", output="sos")
    return sosfiltfilt(sos, audio)


def metricas_reconstrucao(original, recuperado, margem=100):
    original = np.asarray(original)
    recuperado = np.asarray(recuperado)
    if len(original) != len(recuperado):
        raise ValueError("Original e recuperado precisam estar alinhados")
    if len(original) <= 2 * margem:
        margem = 0
    ref = original[margem:-margem] if margem else original
    est = recuperado[margem:-margem] if margem else recuperado
    mse = np.mean((ref - est)**2)
    snr_rec = 10 * np.log10(np.mean(ref**2) / (mse + 1e-15))
    return mse, snr_rec

## 4. Transmissão e recuperação FM

Executamos a cadeia com SNR de entrada de 20 dB e filtro passa-baixas de 1.100 Hz. A normalização do player é apenas para reprodução; as métricas usam as amostras não normalizadas.

In [ ]:
SNR_DB = 20
FC_HZ = FC_REFERENCIA_HZ

tx = modular_fm(mensagem)
rx = canal_awgn(tx, SNR_DB, np.random.default_rng(SEMENTE))
demodulado = demodular_fm(rx)
recuperado = filtrar_audio(demodulado, FC_HZ)
referencia = mensagem[1:]

mse, snr_reconstrucao = metricas_reconstrucao(referencia, recuperado)
print(f"SNR de entrada: {SNR_DB} dB")
print(f"MSE: {mse:.6f} | SNR de reconstrução: {snr_reconstrucao:.2f} dB")
print("Áudio original:")
display(Audio(mensagem, rate=FS, normalize=True))
print("Áudio recuperado:")
display(Audio(recuperado, rate=FS, normalize=True))

## 5. Espectro, waterfall e comparação temporal

In [ ]:
freq, psd = welch(rx, fs=FS, nperseg=min(1024, len(rx)), return_onesided=False)
ordem = np.argsort(freq)

fw, tw, pxx = spectrogram(
    rx, fs=FS, nperseg=512, noverlap=448,
    return_onesided=False, mode="psd"
)
ordem_w = np.argsort(fw)

fig, eixos = plt.subplots(3, 1, figsize=(11, 11), constrained_layout=True)
eixos[0].plot(freq[ordem] / 1000, 10 * np.log10(psd[ordem] + 1e-15))
eixos[0].set(title="Espectro IQ recebido", xlabel="Frequência (kHz)", ylabel="PSD (dB/Hz)")
eixos[0].grid(alpha=0.3)

im = eixos[1].pcolormesh(
    tw, fw[ordem_w] / 1000, 10 * np.log10(pxx[ordem_w, :] + 1e-15),
    shading="auto", cmap="viridis"
)
eixos[1].set(title="Waterfall do sinal FM", xlabel="Tempo (s)", ylabel="Frequência (kHz)")
fig.colorbar(im, ax=eixos[1], label="PSD (dB/Hz)")

n = min(int(0.05 * FS), len(recuperado))
tempo_ms = np.arange(n) / FS * 1000
eixos[2].plot(tempo_ms, referencia[:n], label="Original")
eixos[2].plot(tempo_ms, recuperado[:n], label="Recuperado", alpha=0.7)
eixos[2].set(title="Comparação temporal (50 ms)", xlabel="Tempo (ms)", ylabel="Amplitude")
eixos[2].legend()
eixos[2].grid(alpha=0.3)
plt.show()

## 6. Painel interativo

Altere a SNR de entrada e a frequência de corte para observar a resposta do receptor. O botão gera um novo resultado reproduzível com a semente fixa.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

slider_snr = widgets.IntSlider(value=20, min=0, max=30, step=2, description="SNR (dB)")
slider_fc = widgets.IntSlider(value=1100, min=500, max=4000, step=100, description="Filtro (Hz)")
botao = widgets.Button(description="Executar receptor", button_style="success")
saida = widgets.Output()

def atualizar_painel(_=None):
    with saida:
        clear_output(wait=True)
        recebido = canal_awgn(tx, slider_snr.value, np.random.default_rng(SEMENTE))
        audio = filtrar_audio(demodular_fm(recebido), slider_fc.value)
        mse, snr_rec = metricas_reconstrucao(referencia, audio)
        print(f"MSE: {mse:.6f} | SNR de reconstrução: {snr_rec:.2f} dB")
        f, p = welch(recebido, fs=FS, nperseg=1024, return_onesided=False)
        idx = np.argsort(f)
        fw, tw, s = spectrogram(recebido, fs=FS, nperseg=512, noverlap=448,
                               return_onesided=False, mode="psd")
        idx_w = np.argsort(fw)
        fig, ax = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
        ax[0].plot(f[idx] / 1000, 10 * np.log10(p[idx] + 1e-15))
        ax[0].set(xlabel="Frequência (kHz)", ylabel="PSD (dB/Hz)", title="Espectro")
        ax[0].grid(alpha=0.3)
        im = ax[1].pcolormesh(tw, fw[idx_w] / 1000, 10 * np.log10(s[idx_w, :] + 1e-15),
                              shading="auto", cmap="viridis")
        ax[1].set(xlabel="Tempo (s)", ylabel="Frequência (kHz)", title="Waterfall")
        fig.colorbar(im, ax=ax[1], label="PSD (dB/Hz)")
        plt.show()
        display(Audio(audio, rate=FS, normalize=True))

botao.on_click(atualizar_painel)
display(widgets.VBox([slider_snr, slider_fc, botao, saida]))

## 7. Comparação estatística dos filtros

Para cada SNR de entrada, aplicamos os filtros de 1.100 Hz e 3.000 Hz à **mesma realização de ruído**, repetindo 20 vezes. O ponto central é a SNR calculada a partir do MSE médio; as bandas mostram a dispersão das SNRs individuais, uma medida descritiva, **não um intervalo de confiança**.

Este experimento usa áudio sintético por padrão; os valores não devem ser confundidos com os resultados das gravações de voz documentados no README.

In [ ]:
snrs_entrada = np.arange(0, 31, 2)
cortes = (1100, 3000)
repeticoes = 20
rng = np.random.default_rng(42)
curvas = {fc: {"media": [], "desvio": []} for fc in cortes}

for snr_entrada in snrs_entrada:
    erros = {fc: [] for fc in cortes}
    for _ in range(repeticoes):
        recebido = canal_awgn(tx, snr_entrada, rng)
        base = demodular_fm(recebido)
        for fc in cortes:
            audio = filtrar_audio(base, fc)
            mse, _ = metricas_reconstrucao(referencia, audio)
            erros[fc].append(mse)
    for fc in cortes:
        erros_fc = np.asarray(erros[fc])
        ref = referencia[100:-100]
        potencia_ref = np.mean(ref**2)
        snr_media = 10 * np.log10(potencia_ref / (np.mean(erros_fc) + 1e-15))
        snrs_individuais = 10 * np.log10(potencia_ref / (erros_fc + 1e-15))
        curvas[fc]["media"].append(snr_media)
        curvas[fc]["desvio"].append(np.std(snrs_individuais))

plt.figure(figsize=(10, 5))
for fc in cortes:
    media = np.array(curvas[fc]["media"])
    desvio = np.array(curvas[fc]["desvio"])
    plt.plot(snrs_entrada, media, marker="o", label=f"Filtro {fc} Hz")
    plt.fill_between(snrs_entrada, media - desvio, media + desvio, alpha=0.15)
plt.title("Comparação de filtros — áudio sintético")
plt.xlabel("SNR de entrada (dB)")
plt.ylabel("SNR de reconstrução (dB)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 8. Teste opcional com gravação de voz

Execute as células seguintes **somente se desejar** testar um áudio próprio no Google Colab. O FFmpeg converte formatos como `.ogg`, `.mp3` e `.wav` para WAV mono em 16 kHz. A gravação permanece apenas no ambiente de execução do Colab, salvo se você decidir compartilhá-la. **Não publique saídas de áudio pessoais no GitHub.**

In [ ]:
# CÉLULA OPCIONAL — Execute apenas no Google Colab, quando quiser carregar uma gravação.
# from google.colab import files
# from scipy.io import wavfile
# import subprocess
#
# arquivos = files.upload()
# nome = next(iter(arquivos))
# subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", nome,
#                 "-ar", str(FS), "-ac", "1", "voz_local.wav"], check=True)
# taxa, dados = wavfile.read("voz_local.wav")
# voz_local = dados.astype(float)
# voz_local /= max(np.max(np.abs(voz_local)), 1e-12)
# voz_local *= 0.7
# print(f"Gravação carregada: {len(voz_local)/taxa:.2f} segundos")
# display(Audio(voz_local, rate=taxa))

### Validação opcional com gravação própria

Após executar a célula anterior, descomente e execute a célula abaixo. Ela compara as duas frequências de corte para a gravação carregada, com uma realização de ruído em SNR de entrada de 20 dB.

In [ ]:
# CÉLULA OPCIONAL — Requer `voz_local` da célula anterior.
# tx_voz = modular_fm(voz_local)
# rx_voz = canal_awgn(tx_voz, 20, np.random.default_rng(123))
# base_voz = demodular_fm(rx_voz)
# for fc in (1100, 3000):
#     rec = filtrar_audio(base_voz, fc)
#     mse, snr_rec = metricas_reconstrucao(voz_local[1:], rec)
#     print(f"Filtro {fc} Hz | MSE={mse:.6f} | SNR de reconstrução={snr_rec:.2f} dB")
#     display(Audio(rec, rate=FS, normalize=True))

## 9. Resultados e limitações

No desenvolvimento original, o filtro de **1.100 Hz** foi selecionado por maximizar a SNR de reconstrução da primeira gravação nas condições simuladas. Em uma segunda gravação, com SNR de entrada de 20 dB, foram observados aproximadamente **12,8 dB** (filtro de 1.100 Hz) e **9,8 dB** (filtro de 3.000 Hz). Esses valores são resultados do experimento original e **não são recalculados automaticamente** pelo sinal sintético deste notebook.

O filtro estreito pode reduzir ruído, mas também eliminar componentes importantes para a inteligibilidade. O desempenho depende do áudio, da SNR, da realização de ruído e do modelo. O sistema não representa recepção RF real nem transmissão pelo ar.

**Próximas extensões:** análise de diferentes sinais de voz, pré-ênfase/de-ênfase, estimativas de incerteza e integração futura com hardware SDR.